# Solar Eclipse Shadow on Earth

This notebook computes the next solar eclipse (Sun occulted by the Moon) and visualizes the Moon's shadow sweeping across the Earth's surface over time.

## Setup

In [ ]:
import numpy as np
import plotly.graph_objs as go

from ostk.mathematics.geometry.d3.object import Point as Point3d
from ostk.mathematics.geometry.d3.object import Ray as Ray3d
from ostk.mathematics.geometry.d3.object import Cone
from ostk.mathematics.geometry import Angle as MathAngle

from ostk.physics import Environment
from ostk.physics.unit import Length
from ostk.physics.unit import Angle
from ostk.physics.time import Scale
from ostk.physics.time import Instant
from ostk.physics.time import Duration
from ostk.physics.time import Interval
from ostk.physics.time import DateTime
from ostk.physics.coordinate import Frame
from ostk.physics.coordinate import Position
from ostk.physics.coordinate import Velocity
from ostk.physics.coordinate.spherical import LLA
from ostk.physics.environment.object import Geometry
from ostk.physics.environment.object.celestial import Earth
from ostk.physics.environment.object.celestial import Sun
from ostk.physics.environment.object.celestial import Moon

from ostk.astrodynamics.solver import TemporalConditionSolver
from ostk.astrodynamics.utilities import lla_from_position

---

## Environment

In [ ]:
environment = Environment.default()

earth = environment.access_celestial_object_with_name("Earth")
sun = environment.access_celestial_object_with_name("Sun")
moon = environment.access_celestial_object_with_name("Moon")

## Find the Next Solar Eclipse

We detect a solar eclipse by checking when the **line segment between the Sun and Moon centers** intersects with the **Earth geometry**.
When this intersection exists, the Moon's shadow is falling somewhere on Earth's surface.

We use the `TemporalConditionSolver` to find the precise time intervals.

In [ ]:
search_interval = Interval.closed(
    Instant.parse("2026-08-01 00:00:00", Scale.UTC),
    Instant.parse("2026-09-01 00:00:00", Scale.UTC),
)

solver = TemporalConditionSolver(
    time_step=Duration.minutes(30.0),
    tolerance=Duration.seconds(10.0),
)

time_step = Duration.seconds(30.0)

In [ ]:
def sun_moon_line_intersects_earth(instant: Instant) -> bool:
    sun_pos = sun.get_position_in(Frame.GCRF(), instant).get_coordinates()
    moon_pos = moon.get_position_in(Frame.GCRF(), instant).get_coordinates()

    # Moon must be between Sun and Earth (closer to Sun than Earth is)
    d_sun_moon = np.linalg.norm(moon_pos - sun_pos)
    d_sun_earth = np.linalg.norm(sun_pos)  # Earth is at GCRF origin
    if d_sun_moon > d_sun_earth:
        return False

    direction = (moon_pos - sun_pos) / d_sun_moon
    ray = Ray3d(Point3d(*sun_pos), direction)
    ray_geometry = Geometry(ray, Frame.GCRF())
    earth_geometry = earth.get_geometry_in(Frame.GCRF(), instant)
    return ray_geometry.intersects(earth_geometry)


eclipse_windows = solver.solve(
    condition=sun_moon_line_intersects_earth,
    interval=search_interval,
)

print(f"Found {len(eclipse_windows)} solar eclipse window(s):")
for i, window in enumerate(eclipse_windows):
    print(f"  Eclipse {i + 1}: {window}")

## Compute the Shadow using OSTk Geometry

For each timestep during the eclipse:

1. **Max eclipse path** — Intersect a `Ray` (Sun → Moon) with the Earth geometry to find the shadow center
2. **Disk radii** — Use the celestial object equatorial radii for the Sun and Moon disks (cross-section perpendicular to the shadow axis)
3. **Shadow cones** — From the two disk radii and positions, construct:
   - A **penumbra `Cone`** (external tangent, apex between Sun and Moon) → intersect with Earth for the partial shadow boundary
   - An **umbra `Cone`** (internal tangent, apex past the Moon) → intersect with Earth for the total shadow boundary

In [ ]:
# Use the first eclipse window found
eclipse_interval = eclipse_windows[0]

# Expand slightly to capture partial phases (penumbra extends beyond the central line)
eclipse_start = eclipse_interval.get_start() - Duration.minutes(30.0)
eclipse_end = eclipse_interval.get_end() + Duration.minutes(30.0)
eclipse_window = Interval.closed(eclipse_start, eclipse_end)

print(f"Eclipse window: {eclipse_window.to_string()}")

In [ ]:
def extract_intersection_llas(
    intersection_geometry,
    object_id: int = 0,
) -> list[LLA]:
    if not intersection_geometry.is_defined():
        return []

    object3d = intersection_geometry.access_composite().access_object_at(0)
    points = []

    if object3d.is_line_string():
        points.extend(
            intersection_geometry.access_composite()
            .access_object_at(object_id)
            .as_line_string()
        )
    elif object3d.is_point():
        points.append(
            intersection_geometry.access_composite().access_object_at(object_id)
        )
    elif object3d.is_point_set():
        points.extend(object3d.as_point_set())

    return [
        lla_from_position(Position.meters(point.as_vector(), Frame.ITRF()))
        for point in points
    ]

In [ ]:
# Disk radii from celestial object properties
r_sun = sun.get_equatorial_radius().in_meters()
r_moon = moon.get_equatorial_radius().in_meters()


def compute_shadow_at_instant(instant):
    """
    Compute eclipse shadow footprint using OSTk geometry operations.

    1. Ray (Sun→Moon) ∩ Earth → shadow center
    2. Penumbra/umbra Cones (from celestial radii) ∩ Earth → shadow boundaries
    """
    sun_position = sun.get_position_in(Frame.ITRF(), instant).get_coordinates()
    moon_position = moon.get_position_in(Frame.ITRF(), instant).get_coordinates()
    earth_geometry = earth.get_geometry_in(Frame.ITRF(), instant)

    sun_to_moon = moon_position - sun_position
    d_sun_moon = np.linalg.norm(sun_to_moon)
    axis = sun_to_moon / d_sun_moon

    # Shadow center (Ray ∩ Earth)
    shadow_ray = Ray3d(Point3d(*sun_position), axis)
    shadow_ray_geometry = Geometry(shadow_ray, Frame.ITRF())
    if not shadow_ray_geometry.intersects(earth_geometry):
        return None
    path_intersection = shadow_ray_geometry.intersection_with(earth_geometry)
    center_llas = extract_intersection_llas(path_intersection)
    if not center_llas:
        return None

    # Shadow cones ∩ Earth
    # Penumbra cone: external tangent apex between Sun and Moon
    pen_apex_d = r_sun * d_sun_moon / (r_sun + r_moon)
    penumbra_apex = sun_position + axis * pen_apex_d
    penumbra_half_angle = np.arctan(float(r_sun / pen_apex_d))
    penumbra_cone = Cone(
        Point3d(*penumbra_apex),
        axis,
        MathAngle.radians(penumbra_half_angle),
    )
    penumbra_geometry = Geometry(penumbra_cone, Frame.ITRF())
    penumbra_footprint = penumbra_geometry.intersection_with(earth_geometry)
    penumbra_llas = extract_intersection_llas(penumbra_footprint)

    # Umbra cone: internal tangent apex past the Moon toward Earth
    umb_apex_d = r_sun * d_sun_moon / (r_sun - r_moon)
    umbra_apex = sun_position + axis * umb_apex_d
    umbra_half_angle = np.arctan(float(r_moon / (umb_apex_d - d_sun_moon)))

    # Cone opens back toward Sun; Earth is inside this cone for a total eclipse
    umbra_cone = Cone(
        Point3d(*umbra_apex),
        -axis,
        MathAngle.radians(umbra_half_angle),
    )
    umbra_geometry = Geometry(umbra_cone, Frame.ITRF())
    umbra_footprint = umbra_geometry.intersection_with(earth_geometry)
    umbra_llas = extract_intersection_llas(umbra_footprint, 1)

    return {
        "center_llas": center_llas,
        "penumbra_llas": penumbra_llas,
        "umbra_llas": umbra_llas,
    }

In [ ]:
instants = eclipse_window.generate_grid(time_step)

shadow_frames = []
for instant in instants:
    result = compute_shadow_at_instant(instant)
    if result is not None and result["penumbra_llas"]:
        result["instant"] = instant
        result["time_str"] = str(instant.to_string())
        shadow_frames.append(result)

print(
    f"Computed shadow at {len(shadow_frames)} timesteps (out of {len(instants)} checked)"
)
if shadow_frames:
    print(f"  First: {shadow_frames[0]['time_str']}")
    print(f"  Last:  {shadow_frames[-1]['time_str']}")
    print(f"  Penumbra boundary points: {len(shadow_frames[0]['penumbra_llas'])}")
    print(f"  Umbra boundary points:    {len(shadow_frames[0]['umbra_llas'])}")

---

## Dynamic Shadow Map

An animated map showing the **penumbra** boundary (partial shadow, gray) and **umbra** boundary (total shadow, dark), both computed by intersecting shadow `Cone` geometries with Earth.
The red line traces the ground track of the shadow center.

In [ ]:
frames = []
slider_steps = []

track_lats = []
track_lons = []

for i, s in enumerate(shadow_frames):
    track_lats.append(s["center_lat"])
    track_lons.append(s["center_lon"])

    pen_lats = [p[0] for p in s["penumbra_llas"]]
    pen_lons = [p[1] for p in s["penumbra_llas"]]
    umb_lats = [p[0] for p in s["umbra_llas"]]
    umb_lons = [p[1] for p in s["umbra_llas"]]

    traces = [
        # Penumbra boundary
        go.Scattergeo(
            lat=pen_lats,
            lon=pen_lons,
            mode="lines+markers",
            line=dict(width=1, color="rgba(100,100,100,0.5)"),
            marker=dict(size=3, color="rgba(100,100,100,0.4)"),
            fill="toself",
            fillcolor="rgba(100,100,100,0.12)",
            name="Penumbra",
            showlegend=(i == 0),
        ),
        # Umbra boundary
        go.Scattergeo(
            lat=umb_lats,
            lon=umb_lons,
            mode="lines+markers",
            line=dict(width=2, color="rgba(30,30,30,0.8)"),
            marker=dict(size=3, color="rgba(30,30,30,0.6)"),
            fill="toself",
            fillcolor="rgba(30,30,30,0.35)",
            name="Umbra",
            showlegend=(i == 0),
        ),
        # Shadow center
        go.Scattergeo(
            lat=[s["center_lat"]],
            lon=[s["center_lon"]],
            mode="markers",
            marker=dict(size=8, color="black", symbol="circle"),
            name="Shadow Center",
            showlegend=(i == 0),
        ),
        # Ground track
        go.Scattergeo(
            lat=track_lats[:],
            lon=track_lons[:],
            mode="lines",
            line=dict(width=2, color="red"),
            name="Ground Track",
            showlegend=(i == 0),
        ),
    ]

    frames.append(go.Frame(data=traces, name=str(i)))
    slider_steps.append(
        dict(
            args=[[str(i)], dict(frame=dict(duration=100, redraw=True), mode="immediate")],
            label=s["time_str"][-15:-5],
            method="animate",
        )
    )

fig = go.Figure(
    data=frames[0].data,
    frames=frames,
    layout=go.Layout(
        title=dict(text="Solar Eclipse Shadow — Cone ∩ Earth"),
        geo=dict(
            showland=True,
            landcolor="rgb(243, 243, 243)",
            countrycolor="rgb(204, 204, 204)",
            coastlinecolor="rgb(150, 150, 150)",
            showocean=True,
            oceancolor="rgb(200, 220, 240)",
            showcountries=True,
            projection_type="natural earth",
            showframe=False,
        ),
        height=700,
        width=1100,
        updatemenus=[
            dict(
                type="buttons",
                showactive=False,
                x=0.05,
                y=0.0,
                xanchor="left",
                yanchor="top",
                buttons=[
                    dict(
                        label="▶ Play",
                        method="animate",
                        args=[
                            None,
                            dict(
                                frame=dict(duration=200, redraw=True),
                                fromcurrent=True,
                                mode="immediate",
                            ),
                        ],
                    ),
                    dict(
                        label="⏸ Pause",
                        method="animate",
                        args=[
                            [None],
                            dict(frame=dict(duration=0, redraw=False), mode="immediate"),
                        ],
                    ),
                ],
            )
        ],
        sliders=[
            dict(
                active=0,
                steps=slider_steps,
                x=0.05,
                len=0.9,
                xanchor="left",
                y=-0.05,
                currentvalue=dict(prefix="Time: ", visible=True),
                transition=dict(duration=100),
            )
        ],
    ),
)

fig.show()

---

## Eclipse Summary

In [ ]:
best_frame = max(shadow_frames, key=lambda f: len(f["umbra_llas"]))

print("=" * 60)
print("  SOLAR ECLIPSE SUMMARY")
print("=" * 60)
print(f"  Eclipse interval:  {eclipse_interval.get_start().to_string()}")
print(f"                  →  {eclipse_interval.get_end().to_string()}")
print(f"  Maximum eclipse:   {best_frame['time_str']}")
print(f"    Center:          ({best_frame['center_lat']:.2f}°, {best_frame['center_lon']:.2f}°)")
print(f"    Sun radius:      {r_sun/1e3:.0f} km")
print(f"    Moon radius:     {r_moon/1e3:.1f} km")
print(f"  Ground track:")
print(f"    Start:           ({track_lats[0]:.1f}°, {track_lons[0]:.1f}°)")
print(f"    End:             ({track_lats[-1]:.1f}°, {track_lons[-1]:.1f}°)")
print("=" * 60)

---

## Method Notes

**All geometry operations use OSTk primitives:**

| Step | OSTk Operation | Result |
|------|---------------|--------|
| Shadow center | `Ray(Sun→Moon)` ∩ `Earth.get_geometry_in()` | Intersection point on Earth's surface |
| Disk radii | `sun.get_equatorial_radius()`, `moon.get_equatorial_radius()` | Sun and Moon cross-section radii |
| Penumbra | `Cone(external tangent apex)` ∩ `Earth.get_geometry_in()` | Boundary curve on Earth |
| Umbra | `Cone(internal tangent apex)` ∩ `Earth.get_geometry_in()` | Boundary curve on Earth |

**Shadow cone geometry:**
- **Penumbra cone**: apex where external tangent lines of the Sun and Moon disks meet (between Sun and Moon), opens toward Earth
- **Umbra cone**: apex where internal tangent lines meet (past the Moon toward Earth), opens back toward Sun — Earth sits inside this cone during a total eclipse

---